# Fairness-Aware GPT-2 — QQP Training (Kaggle)

Trains the paraphrase models from the report and produces the Table 2 numbers.
Scope is **QQP only** — that's where the fairness contribution is.

## Before you run anything

Open the panel on the right (**Session options** / **⋮ → Accelerator**) and set:

| Setting | Value |
|---|---|
| **Accelerator** | **GPU P100** (or T4 x2 — only one is used) |
| **Internet** | **On** ← needs phone verification on your Kaggle account |

Without Internet on, `git clone` and the dataset download both fail.

## Why this doesn't use uv

`uv` exists in this project to pin **CPU** torch, so the Streamlit Cloud app
stays under its 1GB memory limit. That's a deployment concern and it fights you
on a training box — `uv run` re-syncs from `uv.lock` and reinstalls the CPU wheel
underneath you.

Kaggle already ships torch with CUDA. So here we skip uv, put `src` on
`PYTHONPATH`, and run the module directly. Nothing to revert, nothing to fight.

## 0. Check the GPU — this is the gate

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

**Must print `cuda: True`** and a GPU name. If not, set the Accelerator in the
right-hand panel and restart the session. Everything below is pointless without it.

## 1. Clone your repo

**Edit `REPO` first.** Kaggle clones from GitHub — anything you haven't pushed
won't be here.

In [ ]:
REPO = "https://github.com/Ricky11234/fairness-aware-gpt2"

In [ ]:
%cd /kaggle/working
!rm -rf fairness-repo
!git clone -q $REPO fairness-repo
%cd fairness-repo

### Verify you got current code

Must print **1 or more**. If it prints `0`, you haven't pushed — GitHub Desktop →
**Push origin**, then re-run the clone cell.

This matters: `identity.py` both generates the CDA counterfactuals *and* computes
the flip rate. Stale code silently corrupts the number you care about.

In [ ]:
!grep -c AMBIGUOUS_PRONOUNS src/fairness_gpt2/identity.py

## 2. Dependencies

Kaggle already has torch (CUDA), transformers, datasets, numpy and tqdm. This
only fills gaps — pip skips anything already satisfied.

**Do not install torch here.** Kaggle's build is correct; replacing it is how
you end up back on CPU.

In [ ]:
!pip install -q "transformers>=4.41" "datasets>=2.19"

## 3. Download QQP

`--train-size 283011` matches the report's pair count (GLUE ships 363,846) and
cuts ~29% off every run. Dev stays at 40,430 — exactly the report's, so the
fairness metrics stay comparable.

In [ ]:
!PYTHONPATH=src python scripts/download_data.py --only qqp --train-size 283011 --skip-test

## 4. Smoke test — do not skip

Two minutes. Catches setup problems before you spend two hours finding them.

In [ ]:
!PYTHONPATH=src python -m fairness_gpt2.train --mode cda_reg --train data/quora-train.csv --dev data/quora-dev.csv --out /tmp/smoke --epochs 1 --train-subset 2000 --eval-subset 1000

### Check three things

1. **`device=cuda`** — the one that matters. If `cpu`, stop.
2. **`dev_acc` ≈ 0.6–0.7** — correct. 1 epoch on 2,000 of 283,011 pairs. You're
   checking that it *runs*, not that it's good.
3. **`fairness_loss` > 0** — if exactly `0.0`, no counterfactuals were generated
   and `cda_reg` has quietly degraded into plain CDA.

## 5. Train `cda_reg` — the model your app deploys

~1.5–2 hours on P100. `--save-half` writes fp16 (~250MB) for Streamlit Cloud's
memory limit.

Kaggle allows up to 12h sessions and won't kill you for idling like Colab does,
but keep the tab open anyway.

In [ ]:
!PYTHONPATH=src python -m fairness_gpt2.train --mode cda_reg --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/cda_reg --epochs 5 --lambda-fair 0.5 --save-half

## 6. Get your results

`results/reproduced/cda_reg.json` is what makes your dashboard show **your**
numbers beside the paper's.

Everything under `/kaggle/working` also appears in the **Output** panel on the
right, where you can download it directly.

In [ ]:
!zip -qr /kaggle/working/reproduced.zip results/reproduced
!cat results/reproduced/cda_reg.json
from IPython.display import FileLink
FileLink("/kaggle/working/reproduced.zip")

Unzip into your project's `results\reproduced\`, commit, push. Your live app
picks it up automatically.

## 7. Push the model to Hugging Face

Needs a **Write** token from huggingface.co/settings/tokens.

Better on Kaggle: **Add-ons → Secrets** → add `HF_TOKEN`, then use the cell below
so your token never appears in the notebook.

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

In [ ]:
HF_USER = "YOUR-HF-USERNAME"   # <-- EDIT THIS

In [ ]:
!PYTHONPATH=src python scripts/push_to_hub.py --ckpt checkpoints/cda_reg --repo $HF_USER/fairness-gpt2-qqp

Then in your Streamlit app → Settings → Secrets:

```toml
MODEL_REPO = "YOUR-HF-USERNAME/fairness-gpt2-qqp"
```

Live predictions turn on.

---

## 8. Optional — the other two models

`cda_reg` alone is enough to deploy. These only add the full Table 2 comparison.
Run one at a time, re-downloading results after each.

In [ ]:
!PYTHONPATH=src python -m fairness_gpt2.train --mode baseline --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/baseline --epochs 5

In [ ]:
!PYTHONPATH=src python -m fairness_gpt2.train --mode cda --train data/quora-train.csv --dev data/quora-dev.csv --out checkpoints/cda --epochs 5

In [ ]:
!zip -qr /kaggle/working/reproduced.zip results/reproduced
from IPython.display import FileLink
FileLink("/kaggle/working/reproduced.zip")

## What to compare against

At **5 epochs**, your report's Table 4 gives a direct comparison:

| Mode | Dev acc | Subgroup gap | Flip rate |
|---|---|---|---|
| CDA | 0.8835 | 0.0433 | 0.0392 |
| CDA + Reg. | 0.8856 | 0.0512 | 0.0296 |

Table 4 has no 5-epoch baseline — compare that against the 10-epoch row
(0.8955 / 0.0365 / 0.0456) and expect lower accuracy.

**Your flip rate will not match.** The report gives substitution *counts*
(60/20/22) and three examples, not the lists — `identity.py` reconstructs them.
Different lexicon, different counterfactuals. Accuracy should land close; flip
rate is the number that moves.